This is code to test making an icechunk index (?) for a series of ROMS history files. Both the history files and the icechunk results will be in kopah buckets.

In [1]:
import warnings
import os
import pandas as pd
import fsspec
import xarray as xr
from pathlib import Path
from time import time

warnings.filterwarnings("ignore", category=UserWarning)

In [2]:
import icechunk
from obstore.store import from_url
from virtualizarr import open_virtual_dataset
from virtualizarr.parsers import HDFParser
from obspec_utils.registry import ObjectStoreRegistry
from obstore.store import S3Store

In [3]:
os.environ['AWS_REQUEST_CHECKSUM_CALCULATION']='when_required'
os.environ['AWS_RESPONSE_CHECKSUM_VALIDATION']='when_required'

In [4]:
# Load credentials
#_ = load_dotenv(f'{os.environ["HOME"]}/dotenv/protocoast.env', override=True)

# Configuration
storage_endpoint = "https://s3.kopah.uw.edu"
storage_bucket = "liveocean-pmacc"
storage_name = 'icechunk-test-2'
bucket_url = f"s3://{storage_bucket}"

# Setup Filesystem
fs = fsspec.filesystem('s3', anon=False, endpoint_url=storage_endpoint, 
                       skip_instance_cache=True, use_listings_cache=False)

In [5]:
# fs.ls(f'{storage_bucket}/LO_roms/cas7_t2_x11b/f2026.01.01') # testing: works fine

In [5]:
# Define Icechunk Storage & Config
storage = icechunk.s3_storage(
    bucket=storage_bucket,
    prefix=f"icechunk/{storage_name}",
    from_env=True,
    endpoint_url=storage_endpoint,
    region='not-used',
    force_path_style=True,
)

config = icechunk.RepositoryConfig.default()
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix=f"{bucket_url}/",
        store=icechunk.s3_store(region="not-used", anonymous=False, s3_compatible=True, 
                                force_path_style=True, endpoint_url=storage_endpoint),
    ),
)

credentials = icechunk.containers_credentials({f"{bucket_url}/": icechunk.s3_credentials(anonymous=False)})

store_obj = S3Store(
    bucket=storage_bucket,
    endpoint=storage_endpoint, # e.g., "https://rustfs.vm.fedcloud.eu:9001"
    region="not-used",
)

registry = ObjectStoreRegistry({bucket_url: store_obj})
parser = HDFParser()

In [6]:
# --- 1. Get Dates from Icechunk Repo (set_repo) ---
try:
    repo = icechunk.Repository.open(storage, config, authorize_virtual_chunk_access=credentials)
    session = repo.readonly_session("main")
    ds = xr.open_zarr(session.store, consolidated=False, chunks={})
    
    if 'ocean_time' in ds.coords:
        # Extract dates as YYYYMMDD strings
        dates = pd.to_datetime(ds.ocean_time.values)# + pd.Timedelta(days=1)
        set_repo = set(dates.strftime('%Y%m%d'))
    else:
        set_repo = set()
        
except Exception as e:
    print(f"Repo access failed or empty ({e}). Assuming set_repo is empty.")
    repo = None
    set_repo = set()

print(f"set_repo: {len(set_repo)} dates found.")

set_repo: 2 dates found.


In [7]:
dates

DatetimeIndex(['2026-01-01 00:00:00', '2026-01-01 01:00:00',
               '2026-01-01 02:00:00', '2026-01-01 03:00:00',
               '2026-01-01 04:00:00', '2026-01-01 05:00:00',
               '2026-01-01 06:00:00', '2026-01-01 07:00:00',
               '2026-01-01 08:00:00', '2026-01-01 09:00:00',
               '2026-01-01 10:00:00', '2026-01-01 11:00:00',
               '2026-01-01 12:00:00', '2026-01-01 13:00:00',
               '2026-01-01 14:00:00', '2026-01-01 15:00:00',
               '2026-01-01 16:00:00', '2026-01-01 17:00:00',
               '2026-01-01 18:00:00', '2026-01-01 19:00:00',
               '2026-01-01 20:00:00', '2026-01-01 21:00:00',
               '2026-01-01 22:00:00', '2026-01-01 23:00:00',
               '2026-01-02 00:00:00'],
              dtype='datetime64[ns]', freq=None)

In [8]:
# --- 2. Get Dates from Cloud Bucket (set_cloud) ---
print("Scanning S3 for liveocean files...")

# day 1
#nos_files = fs.glob(f'{bucket_url}/LO_roms/cas7_t2_x11b/f2026.01.01/ocean_his*.nc')

# day 2
nos_files = fs.glob(f'{bucket_url}/LO_roms/cas7_t2_x11b/f2026.01.02/ocean_his*.nc')
nos_files = nos_files[1:] # drop hour zero because it was done in day 1


nos_urls = []
for f in nos_files:
    nos_path = f's3://{f}'
    nos_urls.append(nos_path)

Scanning S3 for liveocean files...


In [9]:
nos_urls

['s3://liveocean-pmacc/LO_roms/cas7_t2_x11b/f2026.01.02/ocean_his_0002.nc',
 's3://liveocean-pmacc/LO_roms/cas7_t2_x11b/f2026.01.02/ocean_his_0003.nc',
 's3://liveocean-pmacc/LO_roms/cas7_t2_x11b/f2026.01.02/ocean_his_0004.nc',
 's3://liveocean-pmacc/LO_roms/cas7_t2_x11b/f2026.01.02/ocean_his_0005.nc',
 's3://liveocean-pmacc/LO_roms/cas7_t2_x11b/f2026.01.02/ocean_his_0006.nc',
 's3://liveocean-pmacc/LO_roms/cas7_t2_x11b/f2026.01.02/ocean_his_0007.nc',
 's3://liveocean-pmacc/LO_roms/cas7_t2_x11b/f2026.01.02/ocean_his_0008.nc',
 's3://liveocean-pmacc/LO_roms/cas7_t2_x11b/f2026.01.02/ocean_his_0009.nc',
 's3://liveocean-pmacc/LO_roms/cas7_t2_x11b/f2026.01.02/ocean_his_0010.nc',
 's3://liveocean-pmacc/LO_roms/cas7_t2_x11b/f2026.01.02/ocean_his_0011.nc',
 's3://liveocean-pmacc/LO_roms/cas7_t2_x11b/f2026.01.02/ocean_his_0012.nc',
 's3://liveocean-pmacc/LO_roms/cas7_t2_x11b/f2026.01.02/ocean_his_0013.nc',
 's3://liveocean-pmacc/LO_roms/cas7_t2_x11b/f2026.01.02/ocean_his_0014.nc',
 's3://liveo

In [10]:
# --- Process NOS ---

#nos_urls_short = nos_urls[:4]

tt0 = time()

print(f"Virtualizing {len(nos_urls)} NOS files...")

nos_list = [
    open_virtual_dataset(url, parser=parser, registry=registry, loadable_variables=['ocean_time'])
    for url in nos_urls
]
print('created nos_list (%0.1f sec)' % (time()-tt0))
# this is the slow step (100 sec for 25 files)

Virtualizing 24 NOS files...
created nos_list (105.5 sec)


In [11]:
# avoid a warning message
xr.set_options(use_new_combine_kwarg_defaults=True)

combined_nos = xr.concat(
    nos_list, dim="ocean_time", coords="minimal", compat="override", combine_attrs="override"
)

print('done')
# this is fast

done


In [12]:
combined_nos

<xarray.Dataset> Size: 50GB
Dimensions:          (tracer: 14, boundary: 4, s_rho: 30, s_w: 31,
                      eta_rho: 1302, xi_rho: 663, eta_u: 1302, xi_u: 662,
                      eta_v: 1301, xi_v: 663, eta_psi: 1301, xi_psi: 662,
                      ocean_time: 24)
Coordinates:
    s_rho            (s_rho) float64 240B ManifestArray<shape=(30,), dtype=fl...
    s_w              (s_w) float64 248B ManifestArray<shape=(31,), dtype=floa...
    lon_rho          (eta_rho, xi_rho) float64 7MB ManifestArray<shape=(1302,...
    lat_rho          (eta_rho, xi_rho) float64 7MB ManifestArray<shape=(1302,...
    lon_u            (eta_u, xi_u) float64 7MB ManifestArray<shape=(1302, 662...
    lat_u            (eta_u, xi_u) float64 7MB ManifestArray<shape=(1302, 662...
    lon_v            (eta_v, xi_v) float64 7MB ManifestArray<shape=(1301, 663...
    lat_v            (eta_v, xi_v) float64 7MB ManifestArray<shape=(1301, 663...
    lon_psi          (eta_psi, xi_psi) float64 7MB ManifestArray<shape=(1301,...
    lat_psi          (eta_psi, xi_psi) float64 7MB ManifestArray<shape=(1301,...
  * ocean_time       (ocean_time) datetime64[ns] 192B 2026-01-02T01:00:00 ......
Dimensions without coordinates: tracer, boundary, eta_rho, xi_rho, eta_u, xi_u,
                                eta_v, xi_v, eta_psi, xi_psi
Data variables: (12/163)
    ntimes           int32 4B ManifestArray<shape=(), dtype=int32, chunks=()>
    ndtfast          int32 4B ManifestArray<shape=(), dtype=int32, chunks=()>
    dt               float64 8B ManifestArray<shape=(), dtype=float64, chunks...
    dtfast           float64 8B ManifestArray<shape=(), dtype=float64, chunks...
    dstart           float64 8B ManifestArray<shape=(), dtype=float64, chunks...
    shuffle          int32 4B ManifestArray<shape=(), dtype=int32, chunks=()>
    ...               ...
    EminusP          (ocean_time, eta_rho, xi_rho) float32 83MB ManifestArray...
    swrad            (ocean_time, eta_rho, xi_rho) float32 83MB ManifestArray...
    sustr            (ocean_time, eta_u, xi_u) float32 83MB ManifestArray<sha...
    svstr            (ocean_time, eta_v, xi_v) float32 83MB ManifestArray<sha...
    bustr            (ocean_time, eta_u, xi_u) float32 83MB ManifestArray<sha...
    bvstr            (ocean_time, eta_v, xi_v) float32 83MB ManifestArray<sha...
Attributes: (12/43)
    file:              /gscratch/macc/parker/LO_roms/cas7_t2_x11b/f2026.01.02...
    format:            netCDF-4/HDF5 file
    Conventions:       CF-1.4, SGRID-0.3
    type:              ROMS history file
    title:             LiveOcean input file
    var_info:          /gscratch/macc/parker/LO_roms_source_git/ROMS/External...
    ...                ...
    compiler_flags:    -fp-model precise -heap-arrays -ip -O3 -traceback -che...
    tiling:            12x16
    history:           ROMS, Version 4.3, Friday - April 17, 2026 -  5:22:10 PM
    ana_file:          ROMS/Functionals/ana_btflux.h, ROMS/Functionals/ana_st...
    CPP_options:       X11B, ADD_FSOBC, ADD_M2OBC, ANA_BPFLUX, ANA_BSFLUX, AN...
    bio_file:          /gscratch/macc/parker/LO_roms_user/x11b/fennel.h

In [13]:
ds_final = combined_nos

if ds_final is not None:
    # Ensure we have a valid repo object
    # Note I had to delete the existing repo to make this work.
    if repo is None:
        repo = icechunk.Repository.create(storage, config, authorize_virtual_chunk_access=credentials)
        initial_session = repo.writable_session("main")

        # Append
        print(f"Writing {len(ds_final.ocean_time)} time steps to Icechunk...")
        ds_final.virtualize.to_icechunk(initial_session.store)
    
        # Commit
        msg = f"Initialized with forecast data:"# {new_dates[0]} to {new_dates[-1]}"
        initial_session.commit(msg)
        print(f"Commit successful: '{msg}'")
    # Create Writable Session
    else:
        append_session = repo.writable_session("main")

        # Append
        print(f"Appending {len(ds_final.ocean_time)} time steps to Icechunk...")
        ds_final.virtualize.to_icechunk(append_session.store, append_dim="ocean_time")
    
        # Commit
        msg = f"Appended forecast data:"# {new_dates[0]} to {new_dates[-1]}"
        append_session.commit(msg)
        print(f"Commit successful: '{msg}'")

    # Verify History
    history = repo.ancestry(branch="main")
    latest = next(history)
    print(f"Latest Commit [{latest.written_at}]: {latest.message}")
    
else:
    print("Nothing to append.")

Appending 24 time steps to Icechunk...
Commit successful: 'Appended forecast data:'
Latest Commit [2026-07-26 15:17:56.812769+00:00]: Appended forecast data:
